# 03 Recommenders

This notebook contains everything after the EDA. Part 1 sets up the evaluation that every recommender uses. The following parts build and evaluate the recommenders, and the last parts compare them and estimate the business value.

| Part | Content |
|---|---|
| 1 | Evaluation setup |
| 2 | Non-personalized recommenders (random, popular, demographic) |
| 3 | Collaborative filtering |
| 4 | Content-based |
| 5 | Hybrid |
| 6 | Context-aware |
| 7 | Bandits |
| 8 | Final comparison and business value |

The data is the cleaned sample from `01_preprocessing.ipynb`, described in `02_eda.ipynb`.

# Part 1. Evaluation setup

Every recommender is compared on the same split, the same definition of a relevant anime and the same metric functions. This part makes those choices once and checks them.

## 1.1 Libraries

In [1]:
%pip install -q pandas pyarrow scikit-surprise rexmex "seaborn>=0.13" "plotly>=5" funcsigs
%pip install -q --no-deps recmetrics

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


**Why recmetrics is installed without its dependencies.** recmetrics 0.1.5 requires pandas below 2.0 and seaborn below 0.12. The course notebooks work around this by switching Colab to Python 3.11 and installing old versions of pandas, numpy and seaborn [4, p. 1]. That Colab runtime (2025.07) is no longer offered, and current runtimes use Python 3.12 [10], for which pandas 1.5 is not available. Installing recmetrics with `--no-deps` on top of current pandas, seaborn and plotly avoids the conflict. The metric functions themselves only use numpy, pandas, scipy and scikit-learn, which work in current versions, and they return the same values as the examples in the S08 notebook.

In [2]:
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import recmetrics
import sklearn.metrics as metrics
from rexmex.metrics.ranking import mean_average_precision_at_k, mean_average_recall_at_k
from sklearn.model_selection import TimeSeriesSplit
from surprise import AlgoBase, Dataset, Reader, SVD, accuracy
from surprise.model_selection import train_test_split

# Set a concrete seed to make results reproducible
import random
my_seed = 1234
random.seed(my_seed)
np.random.seed(my_seed)

## 1.2 Data, explicit and implicit feedback

In [3]:
interactions_df = pd.read_parquet("../data/processed/interactions.parquet")
users_df = pd.read_parquet("../data/processed/users.parquet")
anime_df = pd.read_parquet("../data/processed/anime.parquet")

# Explicit feedback: rows with a score
ratings_df = interactions_df[interactions_df["score"] > 0].reset_index(drop=True)
print("Explicit ratings:", len(ratings_df))

status_names = {1: "watching", 2: "completed", 3: "on hold", 4: "dropped", 6: "plan to watch"}
print()
print("Average score per status")
print(ratings_df.groupby(ratings_df["status"].map(status_names))["score"].agg(["mean", "count"]).round(2))

Explicit ratings: 977062

Average score per status
           mean   count
status                 
completed  7.66  893971
dropped    4.93   33828
on hold    7.20   22644
watching   7.98   26619


In [4]:
# Explicit relevance: a score of 8 or more (see 02_eda)
relevant_score = 8

# Implicit feedback: 1 for completed or watching, 0 for on hold, dropped and plan to watch
interactions_df["implicit"] = interactions_df["status"].isin([1, 2]).astype(int)
print("Share of interactions labelled 1:", round(interactions_df["implicit"].mean(), 3))

Share of interactions labelled 1: 0.686


**Implicit feedback.** An interaction counts as positive (1) when the anime is completed or being watched. On hold, dropped and plan to watch count as 0.

The reason is that implicit feedback should mark consumption the user was happy with, not every action. A purchase or a full stream is a positive implicit rating, while returning a product is a negative one [2, p. 83]. Turning events into 0 and 1 is a design choice that has to be justified for each dataset [3, p. 16]. The scores in this data support the split. Users who rated an anime they are still watching give it 7.98 on average and completed anime get 7.66. Dropped anime get only 4.93, so dropping is clearly a negative signal. On hold anime get 7.20, below both positive statuses and below the relevance threshold of 8 from 02_eda.

A counterargument is that plan to watch also shows interest, since the user chose to add the anime. But the user has not watched it, so it cannot tell whether they liked it. It is closer to a product view than a purchase, which is labelled 0 in the course example [3, p. 16]. On hold is the less clear case, since its average is still positive. It is labelled 0 because the user stopped watching, and only 2.3% of the rated interactions have this status, so the choice has little effect on the results. With this rule 68.6% of all interactions are positive.

## 1.3 Train and test splits
### Random split

In [5]:
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(ratings_df[["user_id", "anime_id", "score"]], reader)

trainset, testset = train_test_split(data, test_size=0.2, random_state=my_seed)

train_items = {trainset.to_raw_iid(i) for i in trainset.all_items()}
test_df = pd.DataFrame(testset, columns=["user_id", "anime_id", "score"])

print("Train ratings:", trainset.n_ratings, " test ratings:", len(testset))
print("Users in the test set:", test_df["user_id"].nunique(), "of", trainset.n_users)
print("Test anime that are not in the train set:", len(set(test_df["anime_id"]) - train_items))
print()
print("Test ratings per user")
print(test_df.groupby("user_id").size().describe().round(1))

Train ratings: 781649  test ratings: 195413
Users in the test set: 4982 of 5000
Test anime that are not in the train set: 57

Test ratings per user
count    4982.0
mean       39.2
std        46.2
min         1.0
25%        11.0
50%        25.0
75%        50.0
max       592.0
dtype: float64


### Split per user

In [6]:
def split_per_user(ratings_df, test_size=0.2):
    # Each user puts test_size of their own ratings in the test set
    test_df = ratings_df.groupby("user_id", group_keys=False).sample(frac=test_size, random_state=my_seed)
    train_df = ratings_df.drop(test_df.index)
    trainset = Dataset.load_from_df(train_df[["user_id", "anime_id", "score"]], reader).build_full_trainset()
    testset = list(test_df[["user_id", "anime_id", "score"]].itertuples(index=False, name=None))
    return trainset, testset

trainset_per_user, testset_per_user = split_per_user(ratings_df)
test_per_user_df = pd.DataFrame(testset_per_user, columns=["user_id", "anime_id", "score"])

print("Users in the test set:", test_per_user_df["user_id"].nunique())
print("Fewest test ratings for a user:", test_per_user_df.groupby("user_id").size().min())

Users in the test set: 5000
Fewest test ratings for a user: 2


### Cross-validation through time (CVTT)

In [7]:
# Ordering by time and resetting the index is needed, since TimeSeriesSplit is based on the index
ratings_by_time_df = ratings_df.sort_values("last_updated").reset_index(drop=True)
tscv = TimeSeriesSplit(n_splits=5)

rows = []
for fold, (train_index, test_index) in enumerate(tscv.split(ratings_by_time_df), start=1):
    train_fold = ratings_by_time_df.loc[train_index]
    test_fold = ratings_by_time_df.loc[test_index]
    rows.append({
        "fold": fold,
        "train until": train_fold["last_updated"].max().date(),
        "test until": test_fold["last_updated"].max().date(),
        "test users with no train history %": round(100 * (1 - test_fold["user_id"].isin(train_fold["user_id"]).mean()), 1),
        "test ratings on anime not in train %": round(100 * (1 - test_fold["anime_id"].isin(train_fold["anime_id"]).mean()), 1),
        "mean test score": round(test_fold["score"].mean(), 2),
    })
pd.DataFrame(rows)

,fold,train until,test until,test users with no train history %,test ratings on anime not in train %,mean test score
0,1,2010-05-06,2012-06-16,56.2,28.1,7.67
1,2,2012-06-16,2014-01-15,47.7,30.3,7.62
2,3,2014-01-15,2015-06-30,34.9,28.5,7.53
3,4,2015-06-30,2016-10-28,31.3,31.1,7.50
4,5,2016-10-28,2018-05-22,25.0,26.7,7.39


**Which split to use.** The random 80/20 split is the main split for comparing models. The per-user split and CVTT are used as checks on the final candidates.

The random split is the standard method in the course [5, p. 4], and it represents almost everything in the test set. 4,982 of the 5,000 users have test ratings, the median user has 25, and only 57 test anime are missing from training. The same split is used by every notebook through the split in 1.3, so all models are tested on exactly the same ratings.

The per-user split puts 20% of every user's ratings in the test set. It solves the problem that a random split can leave a user with no test data by chance [5, p. 28]. Here it only adds the 18 missing users, and every user gets at least 2 test ratings. Because the gain is small, it is used as a robustness check rather than the main split. It is also one of the advanced evaluation options in the course [9].

CVTT trains on the past and tests on later ratings, which is how the system would work in production [5, p. 27]. The table shows why it cannot be the main split for every model. In the first fold 56.2% of the test users have no history in the training data, and even in the last fold 25.0% have none. Between 26.7% and 31.1% of test ratings are on anime that did not exist in the training period. A collaborative filtering model cannot score these users or anime, so CVTT mixes the cold start problem into every comparison. It is still essential for two parts of the project. The average test score falls from 7.67 to 7.39 across the folds, the same downward drift found in 02_eda, so a random split lets models learn from the future. CVTT therefore measures how the final system and the context-aware models perform on genuinely new data.

The main counterargument to the random split is exactly this leakage of future ratings into training, which makes all results slightly optimistic. The leakage affects every model in the same way, so it does not change which model ranks first. The final candidates are re-tested with CVTT to confirm the choice holds on future data.

## 1.4 Metric functions

`get_top_n` and `ndcg_multiple_users` are the functions from the S10 notebook. `test_ranking_metrics` adds MAP and MAR from rexmex (S08). `top_n_unseen` ranks every anime a user has not rated, which is what a homepage would show.

In [8]:
def get_top_n(predictions, n=10, solve_ties=False):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        if solve_ties:
            user_ratings.sort(key=lambda x: (x[1], x[0]), reverse=True)
        else:
            user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n


def ndcg_multiple_users(relevant_items_all_users, predictions_ranking_all_users, k=5):
    ndcg_list = []
    for i in range(len(relevant_items_all_users)):
        # Skip users with only 1 item in the test data, as no NDCG can be computed
        if len(relevant_items_all_users[i]) > 1:
            ndcg = metrics.ndcg_score([relevant_items_all_users[i]], [predictions_ranking_all_users[i]], k=k)
            ndcg_list.append(ndcg)
    return np.mean(ndcg_list)

In [9]:
def accuracy_metrics(predictions):
    return {"RMSE": accuracy.rmse(predictions, verbose=False),
            "MAE": accuracy.mae(predictions, verbose=False)}


def test_ranking_metrics(predictions, k=10, min_test_items=1):
    # Rank each user's test anime by the predicted score
    df = pd.DataFrame(predictions)
    by_user = df.groupby("uid")[["iid", "r_ui", "est"]].agg(list)
    by_user = by_user[by_user["iid"].str.len() >= min_test_items]

    # MAP and MAR need the relevant anime and the ranking as lists of ids
    relevant_items, rankings = [], []
    for iids, scores, ests in zip(by_user["iid"], by_user["r_ui"], by_user["est"]):
        relevant = [i for i, s in zip(iids, scores) if s >= relevant_score]
        if relevant:
            relevant_items.append(relevant)
            rankings.append([i for _, i in sorted(zip(ests, iids), reverse=True)])

    return {f"NDCG@{k}": ndcg_multiple_users(by_user["r_ui"].tolist(), by_user["est"].tolist(), k=k),
            f"MAP@{k}": mean_average_precision_at_k(relevant_items, rankings, k=k),
            f"MAR@{k}": mean_average_recall_at_k(relevant_items, rankings, k=k),
            "users": len(by_user)}

In [10]:
def eval_users(trainset, n=1000):
    # The same random users for every model
    raw_users = sorted(trainset.to_raw_uid(u) for u in trainset.all_users())
    rng = np.random.default_rng(my_seed)
    return list(rng.choice(raw_users, size=n, replace=False))


def top_n_unseen(algo, trainset, users, n=10, solve_ties=False):
    # Same result as build_anti_testset() + test() + get_top_n(), but one user at a time to save memory
    rng = np.random.default_rng(my_seed)
    all_items = [trainset.to_raw_iid(i) for i in trainset.all_items()]
    top_n = {}
    for uid in users:
        seen = {trainset.to_raw_iid(i) for i, _ in trainset.ur[trainset.to_inner_uid(uid)]}
        scores = [(iid, algo.predict(uid, iid).est) for iid in all_items if iid not in seen]
        if solve_ties:
            scores.sort(key=lambda x: (x[1], x[0]), reverse=True)
        else:
            rng.shuffle(scores)  # ties are broken at random
            scores.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = [iid for iid, _ in scores[:n]]
    return top_n


def homepage_hits(top_n, testset):
    # Relevant test anime among the 10 anime shown to each user
    relevant = defaultdict(set)
    for uid, iid, r in testset:
        if r >= relevant_score:
            relevant[uid].add(iid)
    hits = [len(set(items) & relevant[uid]) for uid, items in top_n.items()]
    return {"hits out of 10": np.mean(hits),
            "users with at least 1 hit %": 100 * np.mean([h > 0 for h in hits])}


genres_df = anime_df.set_index("anime_id")["genre"].fillna("").str.get_dummies(sep=", ")

def beyond_accuracy_metrics(top_n, trainset):
    # Use a model fitted on build_full_trainset(), never on a train and test split
    catalog = [trainset.to_raw_iid(i) for i in trainset.all_items()]
    lists = list(top_n.values())
    return {"coverage %": recmetrics.prediction_coverage(lists, catalog),
            "personalization": recmetrics.personalization(lists),
            "intra-list similarity": recmetrics.intra_list_similarity(lists, genres_df)}

## 1.5 Check of the setup

The setup is checked with three recommenders whose behaviour is known in advance. The random and popular classes are written as in the S10 notebook. The popular one stores the averages in a dictionary instead of a dataframe, which gives the same predictions much faster. Part 2 evaluates and interprets these baselines in full. Here they only check that the evaluation code behaves as expected.

In [11]:
class MyRandomAlgorithm(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def estimate(self, u, i):
        return np.random.normal(loc=self.train_mean, scale=self.train_std)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        ratings = [r for (_, _, r) in self.trainset.all_ratings()]
        self.train_mean = np.mean(ratings)
        self.train_std = np.std(ratings)
        return self


class MyPopularAlgorithm(AlgoBase):
    def __init__(self):
        AlgoBase.__init__(self)

    def estimate(self, u, i):
        # Anime not seen in training get the global mean
        return self.mean_rating_per_item.get(i, self.trainset.global_mean)

    def fit(self, trainset):
        AlgoBase.fit(self, trainset)
        ratings = pd.DataFrame([[i, r] for (_, i, r) in self.trainset.all_ratings()], columns=["item", "rating"])
        self.mean_rating_per_item = ratings.groupby("item")["rating"].mean().to_dict()
        return self


check_models = {
    "Random": MyRandomAlgorithm(),
    "Popular": MyPopularAlgorithm(),
    "SVD (default)": SVD(random_state=my_seed),
}

### Accuracy and ranking of the test items

In [12]:
rows = []
for name, algo in check_models.items():
    algo.fit(trainset)
    predictions = algo.test(testset)
    row = {"model": name}
    row.update(accuracy_metrics(predictions))
    row.update(test_ranking_metrics(predictions))
    row["NDCG@10 (users with 20+ test items)"] = test_ranking_metrics(predictions, min_test_items=20)["NDCG@10"]
    rows.append(row)

pd.DataFrame(rows).set_index("model").round(3)

,RMSE,MAE,NDCG@10,MAP@10,MAR@10,users,NDCG@10 (users with 20+ test items)
model,,,,,,,
Random,2.346,1.859,0.860,0.556,0.387,4982,0.812
Popular,1.552,1.184,0.932,0.784,0.509,4982,0.912
SVD (default),1.247,0.930,0.938,0.800,0.520,4982,0.919


**Ranking metrics on the test items overstate weak models.** The ordering is as expected. SVD is best on every metric, popular comes second and random is last [5, pp. 8-9]. But random still reaches an NDCG@10 of 0.860 and a MAP@10 of 0.556.

Two effects explain this. First, these metrics only rank each user's own test ratings. Many users have few test ratings, and a short list leaves little room for a bad ranking [7, pp. 14-15]. Second, 55.7% of all ratings are 8 or more (02_eda), so even a random order puts relevant anime near the top. The difference between an 8 and a 9 also barely moves NDCG [7, p. 17].

The evidence is in the last column. For users with at least 20 test ratings, the NDCG of random drops from 0.860 to 0.812, while SVD only drops from 0.938 to 0.919. The gap between random and SVD grows from 0.078 to 0.107. Longer test lists make it possible to rank badly, so the metric separates the models better. For this reason every notebook reports NDCG for users with 20 or more test items next to the overall value.

The RMSE results follow the same order. Random has an RMSE of 2.35 because it draws scores around the mean with the full spread of the data. Popular reduces it to 1.55 and SVD to 1.25. One could argue RMSE alone is enough to rank models, but RMSE measures every predicted score, while the homepage only shows the top 10. A model can predict well on average and still rank the top poorly, which is why ranking metrics are needed as well [2, p. 227].

### Top 10 over the whole catalog

In [13]:
users = eval_users(trainset)

rows = []
for name, algo in check_models.items():
    algo.fit(trainset)
    top_n = top_n_unseen(algo, trainset, users, solve_ties=(name == "Popular"))
    row = {"model": name}
    row.update(homepage_hits(top_n, testset))
    rows.append(row)

pd.DataFrame(rows).set_index("model").round(3)

,hits out of 10,users with at least 1 hit %
model,,
Random,0.028,2.7
Popular,0.000,0.0
SVD (default),0.652,39.6


**The homepage metric is the closest to the business question.** It ranks every anime the user has not rated, takes the top 10 and counts how many are relevant anime from the user's test set. SVD finds 0.652 relevant anime per 10 shown, against 0.028 for random. That is about 23 times more, and 39.6% of users get at least one hit against 2.7% with random.

Popular gets 0 hits. Its top 10 consists of anime with an average of 10 from a single rating. Twelve anime in the training data have exactly one rating of 10, so the plain average puts them above every well-known title. Nobody else has rated them, so they never match a test rating. This is a weakness of the plain average, and Part 2 fixes it with a Bayesian average.

The absolute numbers are low on purpose and should not be read as the real hit rate. The metric only counts an anime as a hit if the user rated it in the test set. An anime the user would have loved but never rated counts as a miss. Offline evaluation is therefore biased towards what users already found by themselves [6, pp. 96-97]. The same bias applies to every model, so the ratio between models is meaningful even though the level is too low. This also explains why random gets 0.028 here and 0.16 in 02_eda. The EDA counted all of a user's relevant anime, while here only the 20% in the test set can count.

The same 1,000 users are used for every model. Ranking all unrated anime for 1,000 users means about 6 million predictions per model, which is the size of the anti-testset of the course example (610 users and 9,724 movies) [5, p. 2]. All 5,000 users would take five times longer for the memory-based models in Part 3.

In [14]:
full_trainset = data.build_full_trainset()
full_users = eval_users(full_trainset)

rows = []
for name, algo in check_models.items():
    algo.fit(full_trainset)
    top_n = top_n_unseen(algo, full_trainset, full_users, solve_ties=(name == "Popular"))
    row = {"model": name}
    row.update(beyond_accuracy_metrics(top_n, full_trainset))
    rows.append(row)

pd.DataFrame(rows).set_index("model").round(3)

,coverage %,personalization,intra-list similarity
model,,,
Random,80.52,0.998,0.180
Popular,0.31,0.077,0.141
SVD (default),14.58,0.918,0.269


**The beyond-accuracy metrics behave as theory predicts, which confirms the setup is correct.** Random reaches 80.5% coverage, popular 0.3% and SVD 14.6%. Personalization is 0.998 for random, 0.077 for popular and 0.918 for SVD. This is the expected order. Random should have the highest coverage and personalization, popular the lowest, and collaborative filtering in between [5, p. 20; 8].

These metrics are computed on `build_full_trainset()`, with every anime the user has not rated as candidates. They do not need a test set because they describe what is shown, not whether it is liked. A train and test split would only use the small fraction of known ratings and make all recommenders look alike on coverage [8]. Random cannot reach 100% here because 1,000 users with 10 anime each fill 10,000 slots out of 5,980 anime, and random draws repeat. The expected share of distinct anime is about 81%, which matches the result.

The setup needed one fix to get there. Surprise clips predictions to the rating scale, so the 7.8% of random predictions above 10 all become exactly 10. Each user then has hundreds of anime tied at 10, and a stable sort keeps them in the same catalogue order for everyone. In a first run this gave random a coverage of only 5.3%. Ties are now broken at random, which is the intended default [5, p. 17]. Popular uses `solve_ties=True` instead, so it gives every user the same list [5, p. 18]. Its personalization is still not exactly 0, because anime a user already rated are removed from their list.

Intra-list similarity measures how alike the genres in each list are, so lower means more diverse [4, p. 26]. SVD has the highest value (0.269), which means its lists are the most similar in genre. This fits collaborative filtering, which recommends anime close to what the user already likes. It is a trade-off to watch, not a flaw on its own. Very high diversity, like that of random, is not good either [4, pp. 25-26]. Coverage and personalization also depend on the number of users and the catalogue size [4, pp. 23, 25]. That is why every model is evaluated on the same 1,000 users.

## 1.6 Rules for every part

1. Every part uses `trainset` and `testset` from the random split in 1.3. The per-user split and CVTT are only used as checks.
2. Explicit relevance is `relevant_score = 8`. Implicit relevance is the `implicit` column from 1.2.
3. Report RMSE and MAE, NDCG, MAP and MAR at 10 with `test_ranking_metrics()`, plus NDCG for users with 20 or more test items.
4. Report homepage hits with `top_n_unseen()` on `trainset` and `homepage_hits()`, for the 1,000 users from `eval_users()`.
5. Report coverage, personalization and intra-list similarity with `top_n_unseen()` on `full_trainset` and `beyond_accuracy_metrics()`. Never use a train and test split for these.
6. Set `random_state=my_seed` in every model and split.
7. Compare every model with random, the company's current strategy.

# Part 2. Non-personalized recommenders

To be added.

# Part 3. Collaborative filtering

To be added.

# Part 4. Content-based

To be added.

# Part 5. Hybrid

To be added.

# Part 6. Context-aware

To be added.

# Part 7. Bandits

To be added.

# Part 8. Final comparison and business value

To be added.

# References

[1] Aggarwal, C. C. (2016). Recommender Systems: The Textbook. Springer.

[2] Falk, K. (2019). Practical Recommender Systems. Manning.

[3] S05 notebook. Key data for building recommendation systems.

[4] S08 notebook. Practical exercises on performance metrics.

[5] S10 notebook. Data splitting and evaluation methods.

[6] S20 notebook. Offline vs online evaluation, hyperparameter tuning and A/B testing strategy.

[7] Impact of long-tail into performance metrics notebook.

[8] Class notes, Session 16.

[9] A4 Build your own non-personalized recommender (assignment brief).

[10] Google Colab. Past runtime versions FAQ. https://research.google.com/colaboratory/runtime-version-faq.html